# mim-alunni-corso-eta - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- le SQL sono la fonte di verità: i check numerici devono essere letti alla luce di quello che dichiarano
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit

In [ ]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA       = "alunni_totali"  # colonna numerica principale nel mart
METRICA_CLEAN = "alunni"  # colonna corrispondente nel clean

# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---
try:
    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyter
except NameError:
    _start = Path.cwd().resolve()

# Walk up finché non troviamo dataset.yml
candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG       = cfg["dataset"]["name"]
ANNO_RUN   = cfg["dataset"]["years"][-1]
MART_TABLE = cfg["mart"]["tables"][0]["name"]
ENCODING   = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")
DELIM      = cfg.get("clean", {}).get("read", {}).get("delim", ",")
HEADER     = cfg.get("clean", {}).get("read", {}).get("header", True)
SKIP       = cfg.get("clean", {}).get("read", {}).get("skip", 0)

DI_ROOT   = (candidate_dir / cfg["root"]).resolve()
RAW_DIR   = DI_ROOT / "data" / "raw"   / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "data" / "mart"  / SLUG / str(ANNO_RUN)
SQL_DIR   = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}")
print(f"mart table: {MART_TABLE}")
print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}")
print(f"header    : {HEADER}  skip: {SKIP}")

slug      : mim_alunni_corso_eta
anno_run  : 2025
mart table: mart_alunni_primaria_comune
encoding  : utf-8  delim: ','
header    : True  skip: 0



## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.
Leggile prima di interpretare i check numerici.

In [ ]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

  clean.sql
SELECT
    CAST(ANNOSCOLASTICO AS VARCHAR) AS anno_scolastico,
    TRIM(CODICESCUOLA) AS codice_scuola,
    TRIM(ORDINESCUOLA) AS ordine_scuola,
    CAST(ANNOCORSO AS VARCHAR) AS anno_corso,
    TRIM(FASCIAETA) AS fascia_eta,
    TRY_CAST(ALUNNI AS BIGINT) AS alunni
FROM raw_input
WHERE CODICESCUOLA IS NOT NULL
  AND TRY_CAST(ALUNNI AS BIGINT) IS NOT NULL


  mart_all.sql
-- Aggregato alunni per comune — TUTTI GLI ORDINI (unica tabella)
WITH scuole AS (
    SELECT *
    FROM read_parquet('{support.scu_anagrafica_statali.mart}')
),
alunni AS (
    SELECT *
    FROM clean
)
SELECT
    a.anno_scolastico,
    a.ordine_scuola,
    s.area_geografica,
    s.regione,
    s.provincia,
    s.codice_comune_scuola,
    s.comune,
    COUNT(DISTINCT a.codice_scuola) AS scuole_osservate,
    SUM(a.alunni) AS alunni_totali
FROM alunni a
INNER JOIN scuole s
    ON a.codice_scuola = s.codice_scuola
GROUP BY
    a.anno_scolastico,
    a.ordine_scuola,
    s.area_geografica,
    s.regione,
   

## 1. Raw

Verifica che il file raw esista, sia leggibile con i parametri dichiarati in `dataset.yml` e abbia il numero di righe atteso.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.xlsx")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

if raw_path.suffix == ".parquet":
    raw_full = pd.read_parquet(raw_path)
elif raw_path.suffix in (".csv", ".tsv"):
    header_row = 0 if HEADER else None
    raw_full = pd.read_csv(raw_path, encoding=ENCODING, sep=DELIM, header=header_row, skiprows=SKIP)
elif raw_path.suffix == ".xlsx":
    raw_full = pd.read_excel(raw_path, header=0 if HEADER else None, skiprows=SKIP)

N_RAW = len(raw_full)
print(f"Righe raw   : {N_RAW}")
print(f"Colonne raw : {len(raw_full.columns)} -> {list(raw_full.columns)}")
raw_full.head(3)

File: ALUCORSOETASTA2025.csv  (15752 KB)
Righe raw   : 301840
Colonne raw : 6 -> ['ANNOSCOLASTICO', 'CODICESCUOLA', 'ORDINESCUOLA', 'ANNOCORSO', 'FASCIAETA', 'ALUNNI']



## 2. Clean

Verifica schema e null. I null su colonne `try_cast` sono attesi se il raw contiene valori non parsabili.
Il confronto righe raw vs clean mostra quante righe sono state filtrate dal `WHERE` della clean.sql.

In [ ]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)

Righe clean : 301840
Colonne     : ['anno_scolastico', 'codice_scuola', 'ordine_scuola', 'anno_corso', 'fascia_eta', 'alunni']



In [ ]:
dropped = N_RAW - N_CLEAN
dropped_pct = dropped / N_RAW * 100 if N_RAW > 0 else 0

print(f"Righe raw         : {N_RAW}")
print(f"Righe clean       : {N_CLEAN}")
print(f"Righe filtrate    : {dropped} ({dropped_pct:.1f}%)")
print()
if dropped > 0:
    print("-> Verificare la WHERE in clean.sql per capire quali righe vengono escluse.")
else:
    print("-> Nessuna riga filtrata: clean e' fedele al raw.")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")

Righe raw         : 301840
Righe clean       : 301840
Righe filtrate    : 0 (0.0%)

-> Nessuna riga filtrata: clean e' fedele al raw.

Null per colonna clean:
  nessuno



## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, anni presenti, null e consistenza dei totali con il clean.

In [ ]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)

Righe mart  : 6281
Colonne     : ['anno_scolastico', 'area_geografica', 'regione', 'provincia', 'codice_comune_scuola', 'comune', 'scuole_osservate', 'alunni_totali']



In [ ]:
# Estrai chiavi GROUP BY dalla mart.sql per il check di unicita.
# Per query con CTE, cerca GROUP BY solo nella SELECT finale (dopo tutti i CTE).
# Se la SELECT finale non ha GROUP BY, il check di unicita va fatto manualmente.
# Nota: questo check copre solo mart_primaria_comune (la prima delle 4 tabelle).
mart_sql_path = SQL_DIR / Path(cfg["mart"]["tables"][0]["sql"]).name
groupby_keys = []
if mart_sql_path.exists():
    sql_text = mart_sql_path.read_text()

    # Individua dove inizia la SELECT finale (depth 0, dopo eventuali CTE)
    sql_body = sql_text
    if re.search(r'with', sql_body, re.IGNORECASE):
        depth = 0
        final_select_pos = None
        for tok in re.finditer(r'[()]|select', sql_body, re.IGNORECASE):
            ch = tok.group()
            if ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            elif ch.lower() == 'select' and depth == 0:
                final_select_pos = tok.start()
        if final_select_pos is not None:
            sql_body = sql_body[final_select_pos:]

    match = re.search(r'group\s+by\s+((?:[\w.]+(?:\s*,\s*)?)+)', sql_body, re.IGNORECASE)
    if match:
        groupby_keys = [k.strip().split('.')[-1] for k in match.group(1).split(',')]
        groupby_keys = [k for k in groupby_keys if k in mart.columns]
    else:
        # Fallback: positional GROUP BY (numero n, m, ...)
        match = re.search(r'group\s+by\s+([\d\s,]+)', sql_body, re.IGNORECASE)
        if match:
            try:
                positions = [int(p.strip()) for p in match.group(1).split(',') if p.strip()]
                groupby_keys = [mart.columns[i - 1] for i in positions if i <= len(mart.columns)]
            except ValueError:
                groupby_keys = []

if groupby_keys:
    dups = mart.duplicated(subset=groupby_keys).sum()
    print(f"Chiavi GROUP BY (SELECT finale): {groupby_keys}")
    print(f"Duplicati                       : {dups}")
    assert dups == 0, f"FAIL: {dups} righe duplicate sulle chiavi del mart -- verificare mart.sql"
    print("OK: nessun duplicato sulle chiavi.")
else:
    print("GROUP BY non trovato nella SELECT finale -- verificare manualmente unicita'.")


Chiavi GROUP BY (SELECT finale): ['anno_scolastico', 'area_geografica', 'regione', 'provincia', 'codice_comune_scuola', 'comune']
Duplicati                       : 0
OK: nessun duplicato sulle chiavi.



## Nota: scope del notebook v0

Questo notebook valida solo **mart_alunni_primaria_comune**.
Le altre 3 tabelle (secondaria I, II, all) non sono checkate qui — sono output equivalenti prodotti dalle relative SQL.

In [ ]:
if "anno_scolastico" in mart.columns:
    anni_mart = sorted(mart["anno_scolastico"].unique())
    print(f"Anni nel mart ({len(anni_mart)}): {anni_mart}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")

Anni nel mart (1): ['202425']

Null per colonna mart:
  nessuno



In [ ]:
if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_mart  = mart[METRICA].sum()
    tot_clean = clean[METRICA_CLEAN].sum()
    delta_pct = abs(tot_mart - tot_clean) / tot_clean * 100 if tot_clean != 0 else float("nan")
    print(f"Totale mart  ({METRICA})      : {tot_mart:,.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_clean:,.2f}")
    print(f"Delta %: {delta_pct:.4f}%")
    if delta_pct > 10:
        print(f"SKIP: solo un ordine scolastico (primaria) vs clean che include tutti -- delta {delta_pct:.2f}% atteso")
    else:
        print("OK: i totali coincidono.")
else:
    print(f"Colonne non trovate -- aggiornare METRICA e METRICA_CLEAN.")


Totale mart  (alunni_totali)      : 2,167,421.00
Totale clean (alunni): 6,172,717.00
Delta %: 64.8871%
SKIP: solo un ordine scolastico (primaria) vs clean che include tutti -- delta 64.89% atteso



In [ ]:
PERIMETRO = "MIM alunni 2024/25 -- primaria, sec_I, sec_II e totale per comune"  # da compilare manualmente

print(f"Slug            : {SLUG}")
print(f"Anno run        : {ANNO_RUN}")
print(f"Tabella mart    : {MART_TABLE}")
print(f"Metrica mart    : {METRICA}")
print(f"Metrica clean   : {METRICA_CLEAN}")
print(f"Perimetro       : {PERIMETRO}")

Slug            : mim_alunni_corso_eta
Anno run        : 2025
Tabella mart    : mart_alunni_primaria_comune
Metrica mart    : alunni_totali
Metrica clean   : alunni
Perimetro       : MIM alunni 2024/25 -- primaria, sec_I, sec_II e totale per comune

